In [8]:
import pandas as pd
import html
import re

df = pd.read_csv("../data/fake_job_postings.csv")

sample = df["description"].iloc[17]
print(sample[:400])

Government funding is only available for 16-18 year olds.We have 10 vacancies for career driven, highly motivated individuals living in the Southend-On-Sea area.You will complete a 4 week traineeship before starting your apprenticeship in Customer Service. You will receive full training and support for the training provider.Please apply now and you will be invited to attend the open day.


In [9]:
def clean_text(text):
    # 0. Missing descriptions are NaN (not text), so return empty string
    if not isinstance(text, str):
        return ""

    # 1. Turn HTML codes like &amp; back into normal characters
    text = html.unescape(text)

    # 2. Split glued words: "ControlsLocation" -> "Controls Location"
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)

    # 3. Replace links, emails and phone numbers with placeholder words
    text = re.sub(r"https?://\S+|www\.\S+|#URL_\w+#", " urltoken ", text)
    text = re.sub(r"\S+@\S+|#EMAIL_\w+#", " emailtoken ", text)
    text = re.sub(r"#PHONE_\w+#", " phonetoken ", text)

    # 4. Lowercase
    text = text.lower()

    # 5. Keep only letters and spaces (drops numbers and punctuation)
    text = re.sub(r"[^a-z\s]", " ", text)

    # 6. Squash multiple spaces into one
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [10]:
print("BEFORE:\n", sample[:300])
print("\nAFTER:\n", clean_text(sample)[:300])

BEFORE:
 Government funding is only available for 16-18 year olds.We have 10 vacancies for career driven, highly motivated individuals living in the Southend-On-Sea area.You will complete a 4 week traineeship before starting your apprenticeship in Customer Service. You will receive full training and support 

AFTER:
 government funding is only available for year olds we have vacancies for career driven highly motivated individuals living in the southend on sea area you will complete a week traineeship before starting your apprenticeship in customer service you will receive full training and support for the train


In [11]:
tests = [
    "IC&amp;E Technician &amp; Controls",
    "Earn $500 per week!! Email jobs@quickcash.com NOW",
    "ControlsLocation Deweyville, TX",
    None,
]
for t in tests:
    print(repr(t), "->", repr(clean_text(t)))

'IC&amp;E Technician &amp; Controls' -> 'ic e technician controls'
'Earn $500 per week!! Email jobs@quickcash.com NOW' -> 'earn per week email emailtoken now'
'ControlsLocation Deweyville, TX' -> 'controls location deweyville tx'
None -> ''


In [12]:
import pandas as pd
import html
import re

# Load data
df = pd.read_csv("../data/fake_job_postings.csv")


def clean_text(text):
    """Clean one job posting's text so the model sees consistent words."""
    # Missing values (NaN) are not strings, return empty text
    if not isinstance(text, str):
        return ""

    # 1. Convert HTML codes: &amp; -> &, &nbsp; -> space
    text = html.unescape(text)

    # 2. Remove leftover HTML tags like <br> or <p>
    text = re.sub(r"<[^>]+>", " ", text)

    # 3. Split glued words: "ControlsLocation" -> "Controls Location"
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)

    # 4. Replace contact details and money with placeholder tokens
    text = re.sub(r"https?://\S+|www\.\S+|#URL_\w+#", " urltoken ", text)
    text = re.sub(r"\S+@\S+\.\S+|#EMAIL_\w+#", " emailtoken ", text)
    text = re.sub(r"#PHONE_\w+#", " phonetoken ", text)
    text = re.sub(r"[$₹£€]\s?\d[\d,]*", " moneytoken ", text)

    # 5. Lowercase
    text = text.lower()

    # 6. Keep only letters and spaces
    text = re.sub(r"[^a-z\s]", " ", text)

    # 7. Collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ---- Quick tests ----
tests = [
    "IC&amp;E Technician &amp; Controls",
    "Earn $500 per week!! Email jobs@quickcash.com NOW",
    "ControlsLocation Deweyville, TX",
    None,
]
for t in tests:
    print(repr(t), "->", repr(clean_text(t)))

# ---- Apply to the whole dataset ----
df["clean_description"] = df["description"].apply(clean_text)

# Compare before vs after for one fake posting
row = df[df["fraudulent"] == 1].iloc[0]
print("\nBEFORE:\n", row["description"][:300])
print("\nAFTER:\n", row["clean_description"][:300])

'IC&amp;E Technician &amp; Controls' -> 'ic e technician controls'
'Earn $500 per week!! Email jobs@quickcash.com NOW' -> 'earn moneytoken per week email emailtoken now'
'ControlsLocation Deweyville, TX' -> 'controls location deweyville tx'
None -> ''

BEFORE:
 IC&amp;E Technician | Bakersfield, CA Mt. PosoPrincipal Duties and Responsibilities: Calibrates, tests, maintains, troubleshoots, and installs all power plant instrumentation, control systems and electrical equipment.Performs maintenance on motor control centers, motor operated valves, generators, e

AFTER:
 ic e technician bakersfield ca mt poso principal duties and responsibilities calibrates tests maintains troubleshoots and installs all power plant instrumentation control systems and electrical equipment performs maintenance on motor control centers motor operated valves generators excitation equipm


In [13]:
import numpy as np

# ---- 1. Combine all text columns into one ----
text_cols = ["title", "company_profile", "description", "requirements", "benefits"]

# Join the raw text first (fillna("") turns missing values into empty text)
df["raw_text"] = df[text_cols].fillna("").agg(" ".join, axis=1)

# Then clean the combined text
df["text"] = df["raw_text"].apply(clean_text)


# ---- 2. Metadata features (numbers, not words) ----
def is_missing(col):
    """1 if the column is empty or missing, else 0."""
    return df[col].fillna("").str.strip().eq("").astype(int)

df["missing_company_profile"] = is_missing("company_profile")
df["missing_requirements"]    = is_missing("requirements")
df["missing_benefits"]        = is_missing("benefits")
df["has_salary"]              = 1 - is_missing("salary_range")

# Count contact details in the RAW text (cleaning has already replaced them)
df["n_urls"]   = df["raw_text"].str.count(r"https?://|www\.|#URL_")
df["n_emails"] = df["raw_text"].str.count(r"\S+@\S+\.\S+|#EMAIL_")
df["n_phones"] = df["raw_text"].str.count(r"#PHONE_")

# Scam-style words such as "earn", "cash" or "no experience"
money_pattern = r"(?i)\b(earn|cash|income|weekly pay|no experience|work from home|data entry|immediate start)\b"
df["n_money_words"] = df["raw_text"].str.count(money_pattern)

# Length of the description (log keeps very long posts from dominating)
df["log_desc_len"] = np.log1p(df["description"].fillna("").str.len())


# ---- 3. Check: do these features differ between real and fake? ----
meta_cols = [
    "telecommuting", "has_company_logo", "has_questions",
    "has_salary", "missing_company_profile", "missing_requirements",
    "missing_benefits", "n_urls", "n_emails", "n_phones",
    "n_money_words", "log_desc_len",
]
df.groupby("fraudulent")[meta_cols].mean().T

fraudulent,0,1
telecommuting,0.041319,0.073903
has_company_logo,0.819149,0.326790
has_questions,0.502057,0.288684
has_salary,0.155460,0.257506
missing_company_profile,0.159927,0.677829
missing_requirements,0.149524,0.177829
missing_benefits,0.402962,0.420323
n_urls,0.629540,0.498845
n_emails,0.072940,0.257506
n_phones,0.039908,0.101617


In [14]:
before = len(df)

labels_per_text = df.groupby("text")["fraudulent"].nunique()
conflicts = (labels_per_text > 1).sum()

df = df.drop_duplicates(subset="text", keep="first").reset_index(drop=True)

print("Rows before:", before)
print("Rows after: ", len(df))
print("Removed:    ", before - len(df))
print("Texts labelled both real and fake:", conflicts)
print("Fake % after:", round(df["fraudulent"].mean() * 100, 2))

Rows before: 17880
Rows after:  15656
Removed:     2224
Texts labelled both real and fake: 0
Fake % after: 4.52


In [15]:
keep = ["text"] + meta_cols + ["fraudulent"]

df[keep].to_csv("../data/processed.csv", index=False)

print("Saved:", df[keep].shape)
df[keep].head()

Saved: (15656, 14)


,text,telecommuting,has_company_logo,has_questions,has_salary,missing_company_profile,missing_requirements,missing_benefits,n_urls,n_emails,n_phones,n_money_words,log_desc_len,fraudulent
0,marketing intern we re food and we ve created ...,0,1,0,0,0,0,1,0,0,0,0,6.809039,0
1,customer service cloud video production second...,0,1,0,0,0,0,0,11,0,0,0,7.639161,0
2,commissioning machinery assistant cma valor se...,0,1,0,0,0,0,1,0,0,0,0,5.874931,0
3,account executive washington dc our passion fo...,0,1,0,0,0,0,0,0,0,0,0,7.863651,0
4,bill review manager spot source solutions llc ...,0,1,1,0,0,0,0,0,0,1,0,7.327123,0
